# Contextual Retrieval Implementation

This notebook implements Contextual Retrieval:
1. Load chunks from Phase 1
2. Enrich chunks with contextual prefixes using GPT-4o-mini
3. Create embeddings for contextualized chunks
4. Build FAISS vector store
5. Implement retrieval & generation pipeline
6. Evaluate on 20 QA pairs

**Key Difference**: Chunks are enriched with context before embedding

In [1]:
import os
import json
import numpy as np
import faiss
import asyncio
from openai import OpenAI, AsyncOpenAI

# Initialize OpenAI clients
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
async_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## 1. Load Data

In [2]:
# Load chunks
with open('data/chapter8_chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f'Loaded {len(chunks)} chunks')

Loaded 170 chunks


In [3]:
# Load QA pairs
with open('data/chapter8_qa_pairs.json', 'r', encoding='utf-8') as f:
    qa_pairs = json.load(f)

print(f'Loaded {len(qa_pairs)} QA pairs')

Loaded 20 QA pairs


In [4]:
# Load full document text for context
with open('data/chapter8_text.txt', 'r', encoding='utf-8') as f:
    full_document_text = f.read()

print(f'Loaded full document ({len(full_document_text)} characters)')

Loaded full document (71757 characters)


## 2. Contextual Enrichment

Add contextual prefix to each chunk using GPT-4o-mini

In [5]:
async def enrich_chunk(chunk_text, chunk_page, document_text):
    """Add contextual prefix using GPT-4o-mini (as specified in assignment)"""
    
    prompt = f"""Title: Chapter 8 - Transformers
{document_text[:4000]}

Chunk (Page {chunk_page}):
{chunk_text}

Provide brief context (1-2 sentences) explaining what this chunk discusses in relation to Chapter 8 on Transformers.
Format: "This chunk from Chapter 8 discusses [explanation]."
"""
    
    response = await async_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=150
    )
    
    context = response.choices[0].message.content.strip()
    
    # Return enriched chunk
    return f"{context}\n\n{chunk_text}"

In [6]:
async def enrich_all_chunks(chunks, full_text):
    """Process all chunks with progress tracking"""
    
    # Create tasks for all chunks
    tasks = []
    for chunk in chunks:
        task = enrich_chunk(
            chunk['text'],
            chunk['page'],
            full_text
        )
        tasks.append(task)
    
    # Process with progress tracking
    enriched = []
    for i, task in enumerate(asyncio.as_completed(tasks)):
        result = await task
        enriched.append(result)
        
        # Progress update
        if (i + 1) % 20 == 0 or i + 1 == len(tasks):
            print(f'✓ Enriched {i+1}/{len(tasks)} chunks', end='\r')
    
    print()  # New line
    return enriched

In [7]:
# Run enrichment
print("Enriching all chunks with contextual prefixes...")
print("(This may take 2-5 minutes depending on number of chunks)\n")

enriched_chunks = await enrich_all_chunks(chunks, full_document_text)

print(f'\n✓ Enriched {len(enriched_chunks)} chunks')

Enriching all chunks with contextual prefixes...
(This may take 2-5 minutes depending on number of chunks)

✓ Enriched 170/170 chunks

✓ Enriched 170 chunks


In [8]:
# Preview enriched chunk
print("Original chunk:")
print(chunks[10]['text'][:300])
print("\n" + "="*80 + "\n")
print("Enriched chunk:")
print(enriched_chunks[10][:500])

Original chunk:
stacked blocks. The arrows in the ﬁgure shows how information from the hidden
representations of preceding tokens is incorporated into the transformer block.
Transformer-based language models are complex, and so the details will unfold
over this chapter and the next few chapters. Chapter 7 already d


Enriched chunk:
This chunk from Chapter 8 discusses nucleus sampling, also known as top-p sampling, as an alternative to top-k sampling for generating text with transformers. It highlights the limitation of fixed k values in top-k sampling and explains how nucleus sampling adapts to the varying shapes of probability distributions over words in different contexts.

8.6.2
Nucleus or top-p sampling
One problem with top-k sampling is that k is ﬁxed, but the shape of the probability
distribution over words differs i


In [9]:
# Save enriched chunks
enriched_chunks_data = []
for i, enriched_text in enumerate(enriched_chunks):
    enriched_chunks_data.append({
        'chunk_id': chunks[i]['chunk_id'],
        'text': enriched_text,
        'page': chunks[i]['page'],
        'source': chunks[i]['source']
    })

with open('data/chapter8_contextualized_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(enriched_chunks_data, f, indent=2, ensure_ascii=False)

print('✓ Saved contextualized chunks to data/chapter8_contextualized_chunks.json')

✓ Saved contextualized chunks to data/chapter8_contextualized_chunks.json


## 3. Create Embeddings & FAISS Vector Store

In [10]:
def get_embedding(text):
    """Get embedding from OpenAI text-embedding-3-small"""
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

In [11]:
# Create embeddings for contextualized chunks
print("Creating embeddings for contextualized chunks...")
contextualized_embeddings = []

for i, enriched_text in enumerate(enriched_chunks):
    emb = get_embedding(enriched_text)
    contextualized_embeddings.append(emb)
    
    # Progress update
    if (i + 1) % 50 == 0 or i + 1 == len(enriched_chunks):
        print(f'Embedded {i+1}/{len(enriched_chunks)}', end='\r')

print(f'\n✓ Created {len(contextualized_embeddings)} embeddings')

Creating embeddings for contextualized chunks...
Embedded 170/170
✓ Created 170 embeddings


In [12]:
# Create FAISS index
dimension = len(contextualized_embeddings[0])
contextual_index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
embeddings_array = np.array(contextualized_embeddings).astype('float32')
contextual_index.add(embeddings_array)

print(f'✓ FAISS index created with {contextual_index.ntotal} vectors')

✓ FAISS index created with 170 vectors


In [13]:
# Save FAISS index
os.makedirs('vectorstore/contextual_retrieval', exist_ok=True)
faiss.write_index(contextual_index, 'vectorstore/contextual_retrieval/index.faiss')

print('✓ Saved FAISS index to vectorstore/contextual_retrieval/index.faiss')

✓ Saved FAISS index to vectorstore/contextual_retrieval/index.faiss


## 4. Retrieval & Generation Pipeline

In [14]:
def retrieve_contextual(query, k=3):
    """Retrieve top-k most relevant contextualized chunks"""
    # Get query embedding
    query_embedding = get_embedding(query)
    query_vector = np.array([query_embedding]).astype('float32')
    
    # Search in FAISS index
    distances, indices = contextual_index.search(query_vector, k)
    
    # Format results
    retrieved = []
    for idx, dist in zip(indices[0], distances[0]):
        retrieved.append({
            'chunk': enriched_chunks[idx],
            'page': chunks[idx]['page'],
            'chunk_id': chunks[idx]['chunk_id'],
            'distance': float(dist),
            'similarity': 1 / (1 + float(dist))
        })
    
    return retrieved

In [15]:
def generate_answer(question, retrieved_chunks):
    """Generate answer using GPT-4o-mini"""
    # Format context
    context = '\n\n'.join([
        f"[Page {chunk['page']}]: {chunk['chunk']}"
        for chunk in retrieved_chunks
    ])
    
    # Create prompt
    prompt = f"""You are a helpful assistant answering questions about Chapter 8: Transformers from the Stanford NLP textbook.

Use only the following context to answer the question. If you cannot answer based on the context, say so.

Context:
{context}

Question: {question}

Answer:"""
    
    # Call OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=500
    )
    
    return response.choices[0].message.content

In [16]:
# Test retrieval
test_query = "What is self-attention?"
retrieved_chunks = retrieve_contextual(test_query, k=3)

print(f"Query: {test_query}\n")
print("Retrieved chunks:")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\n{i+1}. Page {chunk['page']} (Similarity: {chunk['similarity']:.3f})")
    print(f"   {chunk['chunk'][:300]}...")

Query: What is self-attention?

Retrieved chunks:

1. Page 19 (Similarity: 0.563)
   This chunk from Chapter 8 discusses the concept of self-attention in transformers, explaining how it enables the model to create contextual representations of tokens by integrating information from surrounding tokens, thereby enhancing the understanding of their relationships over larger spans.

als...

2. Page 20 (Similarity: 0.472)
   This chunk from Chapter 8 discusses the architecture of transformers, specifically focusing on the process of modeling language through a series of transformer blocks that utilize multi-head self-attention to create contextual representations of tokens, ultimately leading to the generation of output...

3. Page 13 (Similarity: 0.469)
   This chunk from Chapter 8 discusses the computation of self-attention in transformers, specifically how embeddings are adjusted using the dimensionality of query and key vectors, and outlines the equations for calculating the self-attent

## 5. Evaluate on All QA Pairs

In [17]:
# Run Contextual Retrieval on all QA pairs
contextual_results = []

print("Running Contextual Retrieval on all QA pairs...\n")

for i, qa in enumerate(qa_pairs):
    # Retrieve relevant chunks (using contextualized index)
    retrieved = retrieve_contextual(qa['question'], k=3)
    
    # Generate answer
    answer = generate_answer(qa['question'], retrieved)
    
    # Store result
    contextual_results.append({
        'question': qa['question'],
        'ground_truth': qa['answer'],
        'contextual_answer': answer,
        'retrieved_chunks': [{
            'text': r['chunk'],
            'page': r['page'],
            'similarity': r['similarity']
        } for r in retrieved]
    })
    
    print(f'✓ Processed {i+1}/{len(qa_pairs)}: {qa["question"][:60]}...', end='\r')

print(f'\n\n✓ Completed {len(contextual_results)} QA pairs')

Running Contextual Retrieval on all QA pairs...

✓ Processed 20/20: What concept distinguishes transformers from feedforward lay...

✓ Completed 20 QA pairs


In [18]:
# Save results
with open('results/contextual_retrieval_responses.json', 'w', encoding='utf-8') as f:
    json.dump(contextual_results, f, indent=2, ensure_ascii=False)

print('✓ Saved results to results/contextual_retrieval_responses.json')

✓ Saved results to results/contextual_retrieval_responses.json


## 6. Preview Results

In [19]:
# Display first 3 results
print("First 3 Q&A Results (Contextual Retrieval):\n")

for i, result in enumerate(contextual_results[:3]):
    print(f"\n{'='*80}")
    print(f"Question {i+1}: {result['question']}")
    print(f"\nGround Truth: {result['ground_truth']}")
    print(f"\nContextual Answer: {result['contextual_answer']}")
    print(f"\nRetrieved from pages: {[c['page'] for c in result['retrieved_chunks']]}")

First 3 Q&A Results (Contextual Retrieval):


Question 1: What is the standard architecture for building large language models?

Ground Truth: The standard architecture for building large language models is the transformer.

Contextual Answer: The standard architecture for building large language models is the transformer.

Retrieved from pages: [13, 26, 2]

Question 2: What is the purpose of transformers in language modeling?

Ground Truth: Transformers are used in language modeling to predict output tokens one by one by conditioning on the prior context.

Contextual Answer: The purpose of transformers in language modeling is to utilize their architecture, which includes components like multi-head attention and embeddings, to generate contextual representations of tokens. This allows them to predict upcoming words in a sequence by processing input tokens through transformer blocks and generating output probabilities over the vocabulary. Transformers also benefit from a wide context wi

---

## Summary

In this notebook, we:
1. ✅ Loaded chunks and QA pairs
2. ✅ Enriched all chunks with contextual prefixes using GPT-4o-mini
3. ✅ Created embeddings for contextualized chunks
4. ✅ Built FAISS vector store for contextual retrieval
5. ✅ Implemented retrieval & generation pipeline
6. ✅ Evaluated on all 20 QA pairs
7. ✅ Saved results to JSON

**Key Insight**: Contextual enrichment adds domain context to each chunk before embedding, potentially improving retrieval relevance.

**Next Steps:**
- Proceed to `04-evaluation.ipynb` to compare Naive RAG vs Contextual Retrieval using ROUGE scores